# 01 - Frame: will this trip crawl?

**Question (one sentence):** at the moment a yellow-taxi trip starts, can we tell from the time and the two zones whether its average speed will end up under 6 mph, so a dispatcher can warn the passenger?

This notebook fixes the target, its labelling decisions and the baseline ladder. See `PROBLEM.md` for the written version.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from src.clean import *
from src.features import *
from src.models import *
from src.evaluate import *
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
FIG = Path("../reports/figures"); FIG.mkdir(parents=True, exist_ok=True)
df, audit = build_dataset()
train, val, test = [df[df.split == s].reset_index(drop=True) for s in ("train", "validation", "test")]


## 1. The target and what can be labelled
Speed = `trip_distance / duration`. A trip is positive if speed < 6 mph. Speed is meaningless for zero distances, zero/negative durations and tiny durations, so those trips cannot be labelled. The table shows the positive rate after each decision, computed on **May only (the training month)** so that no June or July label is looked at while we define the target. A second table on all months follows, only to show the rule choices do not depend on the month.

In [2]:
trips_raw, zones = load_raw()
ladder = target_ladder(trips_raw, zones, month=5)
ladder['positive_rate'] = (ladder.positive_rate*100).round(2)
all_months = target_ladder(trips_raw, zones)
all_months['positive_rate'] = (all_months.positive_rate*100).round(2)
print('MAY ONLY (used for decisions)'); print(ladder.to_string()); print('\nALL MONTHS (comparison only; the steps barely change the rate in any month)'); print(all_months.to_string())

MAY ONLY (used for decisions)
                                                                                    step    rows  positive_rate
0  all rows, naive (speed < 6 incl. zero distance, <=0 duration -> inf/NaN as not crawl)  199999          21.70
1                            raw, speed defined only where duration > 0 and distance > 0  191885          19.76
2                                                       + duration between 1 and 180 min  191180          19.69
3                                                                   + distance >= 0.1 mi  190276          19.31
4                                                                      + speed <= 80 mph  190244          19.31

ALL MONTHS (comparison only; the steps barely change the rate in any month)
                                                                                    step    rows  positive_rate
0  all rows, naive (speed < 6 incl. zero distance, <=0 duration -> inf/NaN as not crawl)  600000          20.

**Reading it.** A naive label counts zero-distance trips (speed 0) as crawls and inflates the rate; the decisions below remove them. Each rule and its reason:

| Decision | Why |
|---|---|
| duration must be > 0 | drop-off not after pickup: speed is undefined |
| duration 1-180 min | under 1 min the speed is rounding noise; over 3 h the meter was left running |
| distance >= 0.1 mi | under ~160 m the distance is not a real trip, and speed 0 would fake a crawl |
| speed <= 80 mph | impossible in NYC, a measurement error |

The full cleaning (negative fares, unknown zones, payment types) is in notebook 02.

## 2. The moment of prediction
At pickup we know the **pickup time, the pickup zone, the drop-off zone** and the **planned route distance**. We do **not** know the duration, drop-off time, measured distance, rate code, payment type, any amount, tip or toll. `src/features.py` holds the `FORBIDDEN` list and notebook 03 tests it.

**Why the planned distance is allowed:** the target is distance/duration. A planned (routed) distance is known when the ride is booked, before any driving happens, so it is a legitimate input; it is not the target's numerator measured after the fact. The files only contain the *measured* distance, so we estimate the plan as the median measured distance of *earlier* trips between the same two zones, learned from May only.It is tested as a candidate feature in notebook 03: it helps on its own, but adds almost nothing once the zone-pair rate is in, so the final model drops it.

## 3. The baseline ladder (scored on June)
1. Always predict 'does not crawl'.
2. A dispatcher rule: weekdays 8-19h, pickup in one of the 20 busiest pickup zones (busiest = most May pickups).
3. A decision tree two levels deep.

In [3]:
M, zh = build_matrix(train, {"val": val})
ytr, yva = train.crawl.values, val.crawl.values
b1 = AlwaysNo().predict_proba(val)[:, 1]
rule = BusyRule().fit(train); b2 = rule.predict_proba(val)[:, 1]
tree2 = make_tree(2).fit(M["train"][ALL_FEATS], ytr); b3 = tree2.predict_proba(M["val"][ALL_FEATS])[:, 1]
rows = {}
for name, p, thr in [("1 always 'no crawl'", b1, 0.5), ("2 busy-hours rule", b2, 0.5), ("3 depth-2 tree", b3, 0.30)]:
    rows[name] = metrics(yva, p, thr)
pd.DataFrame(rows).T.round(3)

,pr_auc,precision,recall,lift@10%,pr_auc_lift,warned_%,base_rate,cost_per_1000
1 always 'no crawl',0.185,0.000,0.000,1.235,1.000,0.000,0.185,556.063
2 busy-hours rule,0.244,0.307,0.480,1.769,1.315,28.957,0.185,490.090
3 depth-2 tree,0.450,0.616,0.437,3.333,2.428,13.133,0.185,363.587


In [4]:
from sklearn.tree import export_text
print(export_text(tree2, feature_names=ALL_FEATS))
print("Rule zones:", sorted(rule.zones_))

|--- pair_rate <= 0.30
|   |--- is_rush <= 0.50
|   |   |--- class: 0
|   |--- is_rush >  0.50
|   |   |--- class: 0
|--- pair_rate >  0.30
|   |--- is_rush <= 0.50
|   |   |--- class: 0
|   |--- is_rush >  0.50
|   |   |--- class: 1

Rule zones: [48, 68, 79, 132, 138, 141, 142, 161, 162, 163, 170, 186, 230, 234, 236, 237, 238, 239, 249, 263]


**Notes.** Baseline 1 has accuracy ~82% yet catches nothing (recall 0), which is why accuracy alone is never reported. Baselines 2 and 3 are the bar the models must clear. The depth-2 tree uses a probability threshold of 0.30 (any value works for 'warn if leaf rate is high'; the real threshold is chosen from costs in notebook 04)Notebook 03 scores the same tree at its best June threshold (precision 0.462, recall 0.607, cost 349)..